# 2.5 Exploring Drug-Target Activity with ChEMBL

Many biological questions require more than knowing that a drug interacts with a particular protein. We may also want to know which compounds have been tested, how their activity was measured, and how strongly they affected the target. these questions matter when comparing candidate compounds, interpreting prior experiments, and deciding which molecules are worth investigating further.

[ChEMBL](https://www.ebi.ac.uk/chembl/) is a public database of bioactive molecules and experimental measurements reported in the scientific literature. Its records connect compounds to biological targets through assays and activity measurements such as IC50, Ki, and EC50 values.

In this notebook, we will use Python to retrieve ChEMBL records through its API and give the selected evidence to a language model to investigate the following question:

> *Which small molecules are best supported by ChEMBL as inhibitors of EGFR?*

Answering this question requires more than simply retrieving every record associated with EGFR. ChEMBL may contain activity measurements from different assay formats, activity types, units, and experimental conditions. We will therefore insepct the records returned by ChEMBL and determine which measurements can be reasonably compared.

Once we have selected and organized the relevant evidence, we will provide it to a language model. The model can help synthesize many structured records into a readable scientific interpretation, compare patterns across compounds, and explain the limitations of the available measurements. Importantly, the language model will not replace the data filtering steps. Its answer will be grounded in the ChEMBL records that we retrieve and prepare in Python.

## 2.5.1 Finding the EGFR Target in ChEMBL

ChEMBL organizes information into linked record types, including targets, assays, molecules, and activities. Before retrieving activity measurements, we first need to identify ChEMBL target record corresponding to human EGFR.

Rather than hard-coding a target identifier, we will search ChEMBL by name and inspect the returned records. This is an important first step because database searches can return multiple related targets, protein complexes, or records from different organisms.

We will use the [ChEMBL REST API documentation](https://www.ebi.ac.uk/chembl/api/data/docs) to query the database directly from Python.

In [1]:
import requests

search_url = "https://www.ebi.ac.uk/chembl/api/data/target/search.json"

target_name = "EGFR"

params = {
    "q": target_name,
}

response = requests.get(search_url, params=params)
response.raise_for_status()

target_results = response.json()["targets"]

for target in target_results[:10]:
    print(target["target_chembl_id"], "|", target["pref_name"], "|", target["organism"], "|", target["target_type"],)

CHEMBL4523747 | EGFR/PPP1CA | Homo sapiens | PROTEIN-PROTEIN INTERACTION
CHEMBL5465557 | CCN2-EGFR | Homo sapiens | PROTEIN-PROTEIN INTERACTION
CHEMBL3608 | Epidermal growth factor receptor | Mus musculus | SINGLE PROTEIN
CHEMBL6193842 | Protein cereblon/Epidermal growth factor receptor | Mus musculus | PROTEIN-PROTEIN INTERACTION
CHEMBL203 | Epidermal growth factor receptor | Homo sapiens | SINGLE PROTEIN
CHEMBL4523680 | Protein cereblon/Epidermal growth factor receptor | Homo sapiens | PROTEIN-PROTEIN INTERACTION
CHEMBL2363049 | Epidermal growth factor receptor | Homo sapiens | PROTEIN FAMILY
CHEMBL3137284 | MER intracellular domain/EGFR extracellular domain chimera | Homo sapiens | CHIMERIC PROTEIN
CHEMBL4523998 | von Hippel-Lindau disease tumor suppressor/Epidermal growth factor receptor | Homo sapiens | PROTEIN-PROTEIN INTERACTION
CHEMBL6193841 | Protein cereblon/Epidermal growth factor receptor | Mus musculus | PROTEIN-PROTEIN INTERACTION


ChEMBL contains several targets associated with the term EGFR, including proteins from different organisms, protein families, and protein complexes. Because our question concerns activity against human EGFR, we will restrict the query to the human single-protein target.

In [2]:
target_url = "https://www.ebi.ac.uk/chembl/api/data/target.json"

params = {
    "pref_name": "Epidermal growth factor receptor",
    "organism": "Homo sapiens",
    "target_type": "SINGLE PROTEIN",
}

response = requests.get(target_url, params=params)
response.raise_for_status()

target_results = response.json()["targets"]

for target in target_results:
    print(target["target_chembl_id"], "|", target["pref_name"], "|", target["organism"], "|", target["target_type"],)

CHEMBL203 | Epidermal growth factor receptor | Homo sapiens | SINGLE PROTEIN


The new query returns the human, single-protein EGFR target, identified in ChEMBL as CHEMBL203.

ChEMBL activity records are linked to targets through these identifiers, so we can now use CHEMBL203 to retrieve experimental measurements reported for EGFR. These records may include several kinds of activity measurements, such as IC50, Ki, or EC50 values, collected under different assay conditions.

Our next step is to retrieve the activity records first, then inspect which fields are available before deciding which measurements are appropriate to compare.

In [25]:
import pandas as pd

activity_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

params = {
    "target_chembl_id": "CHEMBL203",
    "limit": 100,
}

response = requests.get(activity_url, params=params)
response.raise_for_status()

activities = response.json()["activities"]

activity_df = pd.DataFrame(activities)[
    [
        "molecule_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "assay_chembl_id",
        "assay_type",
        "assay_description",
    ]
]

activity_df.head(10)

,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,assay_chembl_id,assay_type,assay_description
0,CHEMBL68920,IC50,=,41.0,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
1,CHEMBL68920,IC50,=,300.0,nM,CHEMBL621151,F,Inhibition of autophosphorylation of human epi...
2,CHEMBL68920,IC50,=,7820.0,nM,CHEMBL615325,F,Inhibition of ligand-induced proliferation in ...
3,CHEMBL69960,IC50,=,170.0,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
4,CHEMBL69960,IC50,=,40.0,nM,CHEMBL621151,F,Inhibition of autophosphorylation of human epi...
5,CHEMBL69960,IC50,=,440.0,nM,CHEMBL615325,F,Inhibition of ligand-induced proliferation in ...
6,CHEMBL137635,IC50,=,9300.0,nM,CHEMBL677833,B,In vitro inhibition of Epidermal growth factor...
7,CHEMBL306988,IC50,=,500000.0,nM,CHEMBL674643,B,Inhibitory concentration of EGF dependent auto...
8,CHEMBL306988,Ki,None,None,nM,CHEMBL675639,B,Inhibition of EGF-dependent epidermal growth f...
9,CHEMBL66879,IC50,=,3000000.0,nM,CHEMBL674643,B,Inhibitory concentration of EGF dependent auto...


Before comparing compound potency, we should first restrict the dataset to measurements that describe the same kind of activity.

For this example, we will focus on IC50 measurements reported in nM with exact (=) relation. Keeping the activity type, units, and relation consistent makes the numerical values more directly comparable and avoids treating censored measurements such as <10 nM or >1000 nM as exact values.

The filtering step is performed in Python so that the criteria are explicit and reproducible. We will then provide the filtered records to the language model for interpretation.

In [26]:
activity_type = "IC50"
activity_units = "nM"

filtered_activity_df = activity_df[
    (activity_df["standard_type"] == activity_type)
    & (activity_df["standard_relation"] == "=")
    & (activity_df["standard_value"].notna())
    & (activity_df["standard_units"] == activity_units)
].copy()

filtered_activity_df["standard_value"] = pd.to_numeric(filtered_activity_df["standard_value"])

filtered_activity_df = filtered_activity_df.sort_values("standard_value")

filtered_activity_df.head(10)

,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,assay_chembl_id,assay_type,assay_description
26,CHEMBL304271,IC50,=,0.45,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
71,CHEMBL67057,IC50,=,4.50,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
89,CHEMBL69629,IC50,=,6.50,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
49,CHEMBL136491,IC50,=,14.00,nM,CHEMBL677833,B,In vitro inhibition of Epidermal growth factor...
67,CHEMBL302552,IC50,=,22.00,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
90,CHEMBL69629,IC50,=,40.00,nM,CHEMBL621151,F,Inhibition of autophosphorylation of human epi...
4,CHEMBL69960,IC50,=,40.00,nM,CHEMBL621151,F,Inhibition of autophosphorylation of human epi...
0,CHEMBL68920,IC50,=,41.00,nM,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphory...
63,CHEMBL137364,IC50,=,64.00,nM,CHEMBL677833,B,In vitro inhibition of Epidermal growth factor...
56,CHEMBL133024,IC50,=,100.00,nM,CHEMBL677833,B,In vitro inhibition of Epidermal growth factor...


## 2.5.2 Interpreting ChEMBL Activity Data with a Language Model

The filtered records are more comparable, but they still come from different assays. Differences in assay format, biological system, substrate concentration, or experimental design can affect the measured IC50. 

Rather than treating every numerical value as directly comparable, we will ask a language model to examine the evidence and identify which measurements can be reasonably compared before drawing conclusions about compound potency.

We will use the same local language model introduced in Section 2.2 and provide the ChEMBL records directly in the prompt.

In [5]:
!pip install -q transformers accelerate ipywidgets requests


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from transformers import pipeline

model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Device set to use mps


In [28]:
activity_text = filtered_activity_df.head(20).to_csv(index=False)

question = "Which small molecules are best supported by ChEMBL as inhibitors of EGFR?"

messages = [
    {
        "role": "system",
        "content": (
            "You are a scientific assistant. Use the provided ChEMBL records as evidence for answering the user's question."
        ),
    },
    {
        "role": "user",
        "content": f"""
            ChEMBL Records:
            {activity_text}
            
            User Question:
            {question}

            Write a concise scientific paragraph and explain your reasoning.
            """,
    },
]

prompt = model.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

response = model(
    prompt,
    max_new_tokens=1024,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None,
)

print(response[0]["generated_text"][len(prompt):])

The ChEMBL database supports several small molecules that appear to be potent inhibitors of the epidermal growth factor receptor (EGFR). The most consistently supported compounds include those with IC50 values ranging from approximately 0.45 nM to 210 nM, which is indicative of strong inhibitory activity against EGFR. For instance, CHEMBL304271, CHEMBL69629, and CHEMBL69960 have been reported to exhibit significant inhibitory effects on EGFR, particularly through their ability to block tyrosine phosphorylation or autophosphorylation. These findings suggest that these molecules may play crucial roles in therapeutic strategies targeting EGFR-related diseases such as cancer. However, it's important to note that while these compounds show promising results, further validation studies would be necessary to confirm their efficacy and safety in clinical applications.


The ChEMBL API gave us structured records describing compounds, assays, and activity measurements. Python was used to identify the relevant target, filter the data, and prepare a consistent set of records for analysis.

The language model then converted those structured records into a concise scientific narrative. This is useful when a database contains many rows of experimentally derived measurements that are easy to process computationally but cumbersome to summarize manually.

The quality of resulting interpretation still depends on the records provided to the model. Careful filtering and clear instructions help ensure that the generated narrative remains grounded in the underlying experimenta evidence.

## 2.5.3 Where to Go Next

This example used ChEMBL to retrieve and summarize bioactivity evidence for small-molecule inhibitors of a single target. The same database can support many other drug-discovery questions.

For example, additional analyses could use ChEMBL to compare activity across related compounds, examine selectivity across multiple targets, identify compounds with both binding and functional evidence, explore ADMET measurements, or assemble curated datasets for predictive modeling.

These workflows follow the same general pattern used throughout this section: retrieve structured evidence with Python, select the records that are relevant to the research question, and use a language model when it can help transform or interpret that evidence.

For further exploration:
- [ChEMBL](https://www.ebi.ac.uk/chembl/)
- [ChEMBL REST API documentation](https://www.ebi.ac.uk/chembl/api/data/docs)
- [ChEMBL Data Web Services](https://chembl.gitbook.io/chembl-interface-documentation/web-services/chembl-data-web-services)